In [ ]:
# Cell 1 - Imports

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.node_parser import SentenceSplitter

print("✓ Imports OK")

In [ ]:
# Cell 2 - Configure models

Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
Settings.llm = Ollama(
    model="deepseek-r1:8b",
    request_timeout=600.0,
    context_window=2048,    # μειώνουμε το context window
    num_output=256
)

print("Models configured")

In [ ]:
# Cell 3 - PDF load

print("Loading papers...")
documents = SimpleDirectoryReader(
    input_dir=".",
    required_exts=[".pdf"]
).load_data()

print(f"Found {len(documents)} chunks from the papers")

In [ ]:
# Cell 3b - Re-index with pymupdf directly
import pymupdf
import os
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

print("Re-indexing with PyMuPDF...")

# Delete existing collection
chroma_client_temp = chromadb.PersistentClient(path="./chroma_db")
chroma_client_temp.delete_collection("astronomy")

# Load PDFs directly with pymupdf
documents = []
for fname in os.listdir("."):
    if fname.endswith(".pdf"):
        print(f"Loading {fname}...")
        doc = pymupdf.open(fname)
        text = ""
        for page in doc:
            text += page.get_text()
        documents.append(Document(text=text, metadata={"file_name": fname}))

print(f"Loaded {len(documents)} documents")

# Chunking
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

# Re-create collection
chroma_collection2 = chroma_client_temp.get_or_create_collection("astronomy")
vector_store2 = ChromaVectorStore(chroma_collection=chroma_collection2)
storage_context2 = StorageContext.from_defaults(vector_store=vector_store2)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context2,
    transformations=[splitter],
    show_progress=True
)

print("✓ Re-indexed successfully!")

In [ ]:
# Cell 4 - Load existing index from ChromaDB (no re-indexing)

print("Loading existing index from ChromaDB...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("astronomy")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=Settings.embed_model
)

print("✓ Index loaded successfully!")

In [ ]:
# Cell 5 - Query with custom prompt
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "You are an astronomy assistant. Use the context below to answer the question.\n"
    "If the context is relevant, use it. Always provide an answer based on your knowledge.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer:"
)

query_engine = index.as_query_engine(
    similarity_top_k=6,
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.3)
    ]
)

query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_prompt}
)

response = query_engine.query(
    "How do black holes relate to dark energy and the expansion of the universe?"
)

print(response)
print("\n--- Sources used ---")
seen = set()
for node in response.source_nodes:
    fname = node.metadata.get('file_name', 'unknown')
    if fname not in seen:
        seen.add(fname)
        print(f"📄 {fname} | score: {node.score:.3f}")